In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# ========== PATHS ========== #
BASE_DIR = "/content/drive/MyDrive/ANNAM.AI Hackathon/soil-classification-part-2/soil_competition-2025"
TRAIN_CSV_PATH = os.path.join(BASE_DIR, "train_labels.csv")  # Verify this path exists!
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train")

# ========== DATA SPLITTING ========== #
# Load labels
train_df = pd.read_csv(TRAIN_CSV_PATH)

# Separate soil (1) and non-soil (0)
soil_df = train_df[train_df['label'] == 1]
non_soil_df = train_df[train_df['label'] == 0]

# Split soil images into train_val (85-15)
train_soil, val_soil = train_test_split(soil_df, test_size=0.15, random_state=42)

# Final sets: train = soil_train + all non_soil | val = soil_val (no non-soil)
train_final = pd.concat([train_soil, non_soil_df])
val_final = val_soil

# ========== DATA GENERATORS ========== #
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=30,
    zoom_range=0.2,
    width_shift_range=0.1,
    brightness_range=[0.8, 1.2]
)

# Simple rescale for validation
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_final,
    directory=TRAIN_IMG_DIR,
    x_col='image_id',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='raw',
    shuffle=True
)

val_gen = val_datagen.flow_from_dataframe(
    dataframe=val_final,
    directory=TRAIN_IMG_DIR,
    x_col='image_id',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='raw',
    shuffle=False  # No shuffle for validation
)

# ========== MODEL & TRAINING ========== #
# Compute class weights (critical for imbalance)
class_weights = {
    0: len(train_final[train_final['label'] == 1]) / len(train_final[train_final['label'] == 0]),
    1: 1.0
}

# Transfer learning setup
base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

model = Model(
    inputs=base_model.input,
    outputs=Dense(1, activation='sigmoid')(base_model.output)
)
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights
)

# ========== PREDICT & SAVE ========== #
# [Add prediction code from previous answer here]


Found 1054 validated image filenames.
Found 183 validated image filenames.


/usr/local/lib/python3.11/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 7 invalid image filename(s) in x_col="image_id". These filename(s) will be ignored.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 1 invalid image filename(s) in x_col="image_id". These filename(s) will be ignored.
  warnings.warn(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 419s 12s/step - accuracy: 0.6734 - loss: 1.7089 - val_accuracy: 0.5847 - val_loss: 0.6452
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 14s 440ms/step - accuracy: 0.8195 - loss: 1.2830 - val_accuracy: 0.6940 - val_loss: 0.5540
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 449ms/step - accuracy: 0.8505 - loss: 0.7645 - val_accuracy: 0.8415 - val_loss: 0.4519
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 452ms/step - accuracy: 0.9091 - loss: 0.6868 - val_accuracy: 0.9126 - val_loss: 0.3903
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 14s 440ms/step - accuracy: 0.9338 - loss: 0.5084 - val_accuracy: 0.9399 - val_loss: 0.3325
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 16s 472ms/step - accuracy: 0.9415 - loss: 0.6510 - val_accuracy: 0.9891 - val_loss: 0.2861
Epoch 7/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 16s 477ms/step - accuracy: 0.9734 - loss: 0.3846 - val_accuracy: 0.9891 - val_loss: 0.2365
Epoch 8/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 438ms/step - accuracy: 0.9739 - loss: 0.4046 - val_accur

In [4]:
import os
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Prepare test data
test_img_dir = os.path.join(BASE_DIR, "test")
test_image_ids = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
test_df = pd.DataFrame({'image_id': test_image_ids})

# Test data generator (use same preprocessing as training)
test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=test_img_dir,
    x_col='image_id',
    y_col=None,
    target_size=(128, 128),
    batch_size=1,
    class_mode=None,
    shuffle=False
)

# Predict
preds = model.predict(test_gen)
pred_labels = (preds > 0.5).astype(int).flatten()  # Threshold at 0.5

# Create and save submission file
submission = pd.DataFrame({
    'image_id': test_df['image_id'],
    'label': pred_labels
})
submission_path = os.path.join(BASE_DIR, 'submission_t2_a.csv')
submission.to_csv(submission_path, index=False)
print(f"Submission file saved successfully at {submission_path}!")


Found 967 validated image filenames.
967/967 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step
Submission file saved successfully at /content/drive/MyDrive/ANNAM.AI Hackathon/soil-classification-part-2/soil_competition-2025/submission_t2_a.csv!
